This program has following functions
1. get_data(type, city, days, alerts) - program generates url based on these parameters and  fetches data for current weather, forecasted weather and alert data
2. get_city() - user has input to type Name of the City
3. get_days() - input number of days for which forecast data is required
4. Then 3 functions extract data from three json objects
    extract_current_data - data from current json
    extract_forecast_data - data from forecast json
    extract_alert_data - data from alert json
5. Output looks like this

City Chicago Current temp 70.0 F and Condition is Partly cloudy
Forecast for the next 15 days is
  Date      Min    Max  Sunrise  Sunset     Condition  
2026-04-24 55.8F 68.2F 05:57 AM 07:42 PM   Patchy rain nearby 
...
Your city has active alerts
Alert number 1
identifier - 4a66ed85-3fa7-11f1-8583-b63eaaffe11e
headline - Flood Warning issued April 23 at 9:10PM CDT until April 25 at 7:00AM CDT by NWS Chicago IL    

In [1]:
import requests
import pandas as pd 
from datetime import datetime

baseurl = "http://api.weatherapi.com/v1"
key = "f67eb958b8fd48839d2190431262104"
city = "Louisville"
type = "current"

In [2]:
def get_data(type, city, days, alerts):
    geturl = f"{baseurl}/{type}.json?key={key}&q={city}"
    if days != 0:
        geturl += f"&days={days}"
    if alerts == 'yes':
        geturl += f"&alerts={alerts}"
    response = requests.get(geturl)
    return response.status_code, response.json()

In [3]:
def get_city():
    city = input("Type the name of the city you want the weather data : ")
    return city

In [4]:
def get_days():
    days = int(input("Type the number of days for forecast data : "))
    return days

In [5]:
def extract_current_data(data):
    curr_temp = data['current']['temp_f']
    curr_cond = data['current']['condition']['text']
    return curr_temp, curr_cond

In [6]:
def extract_forecast_data(data):
    fore_list=[]
    for days in data['forecast']['forecastday']:
        for dayi,dayv in days.items():
            if dayi == 'date':
                date = dayv
            if dayi == 'day':
                for ji, jv in dayv.items():
                    if ji == 'maxtemp_f':
                        max_temp = jv
                    if ji == 'mintemp_f':
                        min_temp = jv
                    if ji == 'condition':
                        cond = jv['text']
            if dayi == 'astro':
                sunrise = dayv['sunrise']
                sunset = dayv['sunset']

        fore_dict = {}
        fore_dict['date']=date
        fore_dict['max_temp']=max_temp
        fore_dict['min_temp']=min_temp
        fore_dict['cond']=cond
        fore_dict['sunrise']=sunrise
        fore_dict['sunset']=sunset
        fore_list.append(fore_dict)
    return fore_list

In [7]:
def extract_alert_data(al_data):
    active_alerts = al_data['alerts']
    return active_alerts

In [8]:
def print_output(city,foredays,curr_temp, curr_cond,fore_list,fc_data,active_alerts):
    print ("\n")
    print (f"City {city} Current temp {curr_temp} F and Condition is {curr_cond}")
    print ("\n")
    print (f"Forecast for the next {foredays} days is")

    # print (f"  Date      Min    Max  Sunrise  Sunset     Condition  ")
    # for i in fore_list:
    #     print (f"{i['date']} {i['min_temp']}F {i['max_temp']}F {i['sunrise']} {i['sunset']}   {i['cond']} ")
    df = pd.DataFrame(fore_list)
    print(df)
    print ("\n")

    if len(active_alerts['alert']) == 0:
        print ("No active alerts for your city")
    else:
        # df1 = pd.DataFrame(active_alerts['alert'])
        # print(df1)

        counter = 0
        for i in active_alerts['alert']:
            counter +=1
            print ("Your city has active alerts")
            print (f"Alert number {counter}")
            for items, val in i.items():
                print (f"{items} - {val}")
            print ("\n")

In [9]:
city = get_city()
type = "current"
days = 0
alerts = ''
curr_status, curr_data = get_data(type, city,days,alerts)
if curr_status == 200:
    curr_temp, curr_cond = extract_current_data(curr_data)

    type = "forecast"
    foredays = get_days()
    fc_status, fc_data = get_data(type, city,foredays,alerts)
    if fc_status == 200:
        fore_list= extract_forecast_data(fc_data)

        type = "alerts"
        alerts = 'yes'
        al_status, al_data = get_data(type, city,days,alerts)
        if al_status == 200:
            active_alerts = extract_alert_data(al_data)
            print_output(city,foredays,curr_temp, curr_cond,fore_list,fc_data,active_alerts)
        else:
            print (f"Something went wrong while retrieving alerts, we hit error {al_status}")
    else:
        print (f"Something went wrong while retrieving forecast, we hit error {fc_status}")
else:
    print (f"Something went wrong, we hit error {curr_status}")



City Chicago Current temp 68.5 F and Condition is Overcast


Forecast for the next 15 days is
          date  max_temp  min_temp                cond   sunrise    sunset
0   2026-04-24      68.2      55.8  Patchy rain nearby  05:57 AM  07:42 PM
1   2026-04-25      52.7      44.9               Sunny  05:55 AM  07:43 PM
2   2026-04-26      59.4      45.3               Sunny  05:54 AM  07:44 PM
3   2026-04-27      63.8      51.9          Heavy rain  05:52 AM  07:45 PM
4   2026-04-28      61.4      51.4  Patchy rain nearby  05:51 AM  07:46 PM
5   2026-04-29      58.2      49.1               Sunny  05:50 AM  07:47 PM
6   2026-04-30      50.3      47.5  Patchy rain nearby  05:48 AM  07:48 PM
7   2026-05-01      55.9      45.7               Sunny  05:47 AM  07:50 PM
8   2026-05-02      59.6      46.6               Sunny  05:46 AM  07:51 PM
9   2026-05-03      62.4      50.8               Sunny  05:44 AM  07:52 PM
10  2026-05-04      54.0      46.7       Partly Cloudy  05:43 AM  07:53 PM
11  